In [20]:
import os
import sys
import session_info
import geopandas as gpd
import numpy as np
import scipy
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import mpl_toolkits
import rtree
from shapely.geometry import Polygon

"""
Reads  Counties 2023 census data
Only uses lower 48 states
Updates percison and projection
Calculates Polsby Popper and Reock, Reock X for each county


"""

Shapepath = "Census data/"
Dataoutpath ="Process data/"

Projection = "ESRI:102008"
#session_info.show()
print( "Libraries loaded")

Libraries loaded


In [21]:
def Border_clip( Geodf, Borderdf):
    """
    Takes a GeoPandas dataframe and clips is borders to the Borders in Borderdf
    Assumes GeoPandas dataframe has rows delineated by GEOID
    Cleans up any non polygon residuals
    Removes any remaining with less than 100,000 sq meters in area
    """
    Districts_clipped = gpd.clip( Geodf, Borderdf, keep_geom_type=False).copy()
    Allgeoms = Districts_clipped.explode()
    Allpolys = Allgeoms[ Allgeoms.geometry.geom_type=="Polygon"].copy()
    Allpolys['Area'] = Allpolys.area
    Allpolys = Allpolys[ Allpolys['Area'] >100000]
    DistrictsM = Allpolys.dissolve(by='GEOID', method='unary').reset_index()
    DistrictsM.drop(['Area'], axis=1, inplace=True)
    return( DistrictsM)

print( "Border_clip function defined")
    

Border_clip function defined


In [22]:
def Geo_check( Geodf ) :
    """
    Analyze the composition of the geo data frame
    Looks for differing geometry types
    """
    ### Count types of geometries found pre clipping
    geometry_types = {'Point': 0, 'LineString': 0, 'Polygon': 0, 'MultiPolygon': 0, 'GeometryCollection':0}

    for geom in Geodf['geometry']:
        geometry_types[geom.geom_type] += 1 

    print("Points:", geometry_types['Point'])
    print("Lines:", geometry_types['LineString'])
    print("Polygons:", geometry_types['Polygon'])
    print("MultiPolygons:", geometry_types['MultiPolygon'])
    print("GeometryCollections:", geometry_types['GeometryCollection'])
    print("Total Polys:", len( Geodf.explode()))
    print( "Total rows:", len(Geodf) )
    print( "Min area polygon:", int( min( Geodf.explode().area)))
    return
    
print( "Geo_check function defined")

Geo_check function defined


In [4]:
def Polsby(Geometry):
    """
    Calculates the Polsby Poppper compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Uses only the exterior ring of each polgon
    Assumes coordinates have already been projected     
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The PolsbyPopper score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    #Darea = Geos["geometry"].area
    Geos = Geos.explode( )
    Geos['geometry'] = Geos['geometry'].apply(lambda p: Polygon(p.exterior.coords)) ## make polygon out of exterior corrdinates
    Geos['Perimeter2'] = Geos["geometry"].length**2
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["PolsbyI"] = (Geos["Area"] * 4 * 3.14159) /  Geos["Perimeter2"] 
    Geos["PolsbyW"] = Geos["PolsbyI"] * Geos["Area"] / Darea
    Polsby = Geos["PolsbyW"].sum() 
    
    return Polsby
print( "Function defined")   

Function defined


In [23]:
def Reock(Geometry):
    """
    Calculates the Reock compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Assumes coordinates have already been projected and precison set    
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The Reock score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    Geos = Geos.explode( )
    Geos['Bounding_circle_area'] = Geos["geometry"].minimum_bounding_circle().area
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["ReockI"] = Geos["Area"] /  Geos["Bounding_circle_area"] 
    Geos["ReockW"] = Geos["ReockI"] * Geos["Area"] / Darea
    Reock = Geos["ReockW"].sum() 
    
    return Reock

def ReockX( District_geometry, State_geometry):
    """
    Calculates the Reock compactness ratio of a given geometry.
    Does not consider the area of minimum bounding circle outside of state lines
    Uses an area weighted average if more than one polygon.
    Assumes coordinates have already been projected and precison set    
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The Reock score
    """
    Geos = gpd.GeoDataFrame( geometry=[District_geometry], crs=Projection)
    Geos = Geos.explode( )
    Geos['Bounding_circle_area'] = Geos["geometry"].minimum_bounding_circle().intersection(State_geometry).area
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["ReockI"] = Geos["Area"] /  Geos["Bounding_circle_area"] 
    Geos["ReockW"] = Geos["ReockI"] * Geos["Area"] / Darea
    ReockX = Geos["ReockW"].sum() 
    
    return ReockX

print( "Reock function defined")  

Reock function defined


In [24]:
#### Calculate state metrics from district unions
def Calc_STdata( Districts ):
    STdata = pd.DataFrame(columns=['STATEFP', 'STarea', 'STperim','STpolsby','STreock','STgeometry'])
    States = Districts[ 'STATEFP'].unique()
    
    for STFP in States :
        STdists = Districts[ Districts['STATEFP'] == STFP]
        State = STdists.geometry.union_all()
        Sarea = int( State.area )
        Sperim = int( State.length )
        Spolsby =  Polsby(State)
        Sreock = Reock(State)
        STdata.loc[len(STdata)] = { 'STATEFP':STFP, 'STarea':Sarea, 'STperim':Sperim, 'STpolsby':Spolsby, 'STreock':Sreock, 'STgeometry':State }

    return( STdata )
#Districts = pd.merge( Districts, STdata, on='STATEFP', how='left')

print( 'Defined state data calc function')


Defined state data calc function


In [25]:
##### Load US boundary file for clipping
File = 'cb_2023_us_nation_20m'
Shapedatafile = Shapepath + File + ".shp"
RawShapes = gpd.read_file( Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Shapes = gpd.clip( RawShapes, USbox)
#Shapes.drop( ['REGIONCE','NAMELSAD', 'LSAD', 'ALAND','AWATER'] , axis=1, inplace=True)
Shapes.to_crs( Projection, inplace=True)
Shapes.geometry = Shapes.geometry.set_precision( 0.01 )
#Shapes.plot()
print( "Loaded ", File," with ", len( Shapes.explode()), " polygons")
#Shapes.head()
US48 = Shapes.copy()

Loaded  cb_2023_us_nation_20m  with  23  polygons


In [26]:
########### Load county shape data
File = 'tl_2023_us_county'
Shapedatafile = Shapepath + File + ".shp"
RawCounties = gpd.read_file( Shapedatafile)
RawCounties.drop( ['NAMELSAD','GEOIDFQ', 'LSAD','CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND','AWATER', 'INTPTLAT', 'INTPTLON'] , axis=1, inplace=True)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Counties = gpd.clip( RawCounties, USbox)
Counties.to_crs( Projection, inplace=True)
Counties.geometry = Counties.geometry.set_precision(0.01)
### Rename GEOIDs that are independent city with dupe county name
Change_name_geoid = ["51620","51770","51760","51600","29510","24510"]  
for geoid in Change_name_geoid:
    Counties.loc[Counties['GEOID'] == geoid, 'NAME'] += '_City'  ### appends _city to each county name

print(  File +" loaded ", len(Counties)," rows of data")
#Counties.plot()
#Counties.head()
#Counties.tail()


tl_2023_us_county loaded  3109  rows of data


In [33]:
### Clip borders to overall US
Districts = Border_clip( Counties, US48 )
#print( 'Number counties post clipping', len(Districts))
#Geo_check( Districts )
print('done clipping')

done clipping


In [34]:
### Calc Reock data
STATE_data = Calc_STdata( Districts )
Districts = pd.merge( Districts, STATE_data, on='STATEFP', how='left')
Districts['Reock'] = Districts.geometry.apply( Reock )
Districts['ReockX'] = Districts.apply(lambda row: ReockX(row.geometry, row.STgeometry), axis=1 )
Districts.drop( columns=['STarea', 'STperim', 'STpolsby','STreock','STgeometry'], axis=1, inplace=True)
print('Done with Reock calcs')
#Districts.head()

Done with Reock calcs


In [35]:
#### Calc county Polaby metrics metrics
Districts['PolsbyW'] = Districts.geometry.apply( Polsby )
print( 'Done with county polsby metrics')
#Districts.head()

Done with county polsby metrics


In [39]:
Column_order = [ 'GEOID', 'STATEFP', 'COUNTYFP', 'COUNTYNS','NAME', 'PolsbyW', 'Reock', 'ReockX' ,'geometry' ]
Districts = Districts.reindex(columns=Column_order)
Shapeoutfile = Dataoutpath + 'County2023.shp'
Districts.to_file( Shapeoutfile )
print( 'Done writing ', len( Districts ), 'rows to ', Shapeoutfile)
#Districts.plot()
#Geo_check( Districts )
CountyData = pd.DataFrame(Districts.drop(columns='geometry'))
CSVoutfile = Dataoutpath + 'County2023.csv'
CountyData.to_csv( CSVoutfile, index=False)
print('Wrote csv file')
#Districts.head()

Done writing  3109 rows to  Process data/County2023.shp
Wrote csv file
